## Full SigLip implementation following [Umar Jamil's tutorial](https://www.youtube.com/watch?v=vAmKB7iPkWw) 


* now that we hace CLIP from scratch in the previous week, lets inspect the suggested reading tutorial and do a deep dive on SigLip
* full notebook is here: [Part 1: Vision Encoders (Umar Jamil)](../suggested_reading/multimodal_lm_from_scratch_umar_jamil/part1_vision_encoders.ipynb)
* The notebook follows a tutorial where we build PaliGemma from scratch but this is only the SigLip part of PaliGemma. 
* We can import classes and take a look at the comments to understand.

In [5]:
import sys      
sys.path.append('../suggested_reading/multimodal_lm_from_scratch_umar_jamil')                                                    
                                                                                                                                
from modeling_siglip import (SiglipVisionConfig,
                            SiglipVisionEmbeddings,                                                                             
                            SiglipAttention,
                            SiglipMLP,
                            SiglipEncoderLayer,
                            SiglipEncoder,                                                                                      
                            SiglipVisionTransformer,
                            SiglipVisionModel)                                                                                  
                                                                                               
                                                                                                                                

## we have our embeddings and patches creation:

In [6]:
# Print the source code                                                                                                          
import inspect  
print(inspect.getsource(SiglipVisionEmbeddings))

class SiglipVisionEmbeddings(nn.Module):
    ## equivalent to PatchEmbedding in vit.py
    def __init__(self, config: SiglipVisionConfig):
        super().__init__()
        self.config = config
        self.embed_dim = config.hidden_size
        self.image_size = config.image_size
        self.patch_size = config.patch_size

        self.patch_embedding = nn.Conv2d(
            in_channels=3,
            out_channels=self.embed_dim,
            kernel_size=self.patch_size,
            stride=self.patch_size,  # no overlap
            padding='valid',  # indicates no padding necessary
        )

        self.num_patches = int((self.image_size**2) / (self.patch_size**2))
        self.num_positions = self.num_patches
        # In our vit.py we used self.pos_embedding = nn.Parameter(torch.randn(1, self.num_patches + 1, num_hiddens))
        # these do the same thing, they create a learnable matrix of positional vectors.
        # under the hood, nn.Embedding is a wrapper for nn.Parameter 

## the components of the encoder:

In [7]:
# the MLP for the feedforward
print(inspect.getsource(SiglipMLP))

class SiglipMLP(nn.Module):
    def __init__(self, config: SiglipVisionConfig):
        super().__init__()
        self.config = config
        self.fc1 = nn.Linear(config.hidden_size, config.intermediate_size)
        self.fc2 = nn.Linear(config.intermediate_size, config.hidden_size)

    def forward(self, hidden_states: torch.Tensor) -> torch.Tensor:
        # [Batch_Size, Num_Patches, Embed_Dim] -> [Batch_Size, Num_Patches, Intermediate_Size]
        hidden_states = self.fc1(hidden_states)
        hidden_states = nn.functional.gelu(hidden_states, approximate='tanh')
        # [Batch_Size, Num_Patches, Intermediate_Size] -> [Batch_Size, Num_Patches, Embed_Dim]
        hidden_states = self.fc2(hidden_states)

        return hidden_states



In [8]:
# the attention caluclator
print(inspect.getsource(SiglipAttention))

class SiglipAttention(nn.Module):
    # no causal mask like language models.
    # start with a sequence of patches, each represented by a 1x1024 vector.
    # each patch is from a group of pixels.
    # the resulting attention mask now has information about the patches relationship to other patches.
    # in language, we contextualize the token against all the tokens that came _before_ it. slight difference than in vit.
    # we use causal mask for next-token prediction task. transformer lets us do that in parallel. hence the causal mask.
    def __init__(self, config: SiglipVisionConfig):
        super().__init__()
        self.config = config
        self.embed_dim = config.hidden_size
        self.num_heads = config.num_attention_heads
        self.head_dim = self.embed_dim // self.num_heads
        self.scale = self.head_dim ** -0.5  # 1/√d_k divisor as a multiplier for efficiency.
        self.dropout = config.attention_dropout

        self.q_proj = nn.Linear(self.embed_dim, sel

In [9]:
# a full encoder layer
print(inspect.getsource(SiglipEncoderLayer))

class SiglipEncoderLayer(nn.Module):
    # this is the same as TransformerEncoder in our vit.py
    # sequence to sequence model here
    # input is embeddings of patches flattened. with attention mechanism, we contextualize these embeddings.
    # each layer will have layernorm --> MHA --> LayerNorm --> FFN.
    def __init__(self, config: SiglipVisionConfig):
        super().__init__()
        self.embed_dim = config.hidden_size
        self.self_attn = SiglipAttention(config)
        self.layer_norm1 = nn.LayerNorm(self.embed_dim, eps=config.layer_norm_eps)
        self.mlp = SiglipMLP(config)
        self.layer_norm2 = nn.LayerNorm(self.embed_dim, eps=config.layer_norm_eps)

    def forward(self, hidden_states: torch.Tensor) -> torch.Tensor:
        # residual [Batch_Size, Num_Patches, Embed_Dim]
        residual = hidden_states
        hidden_states = self.layer_norm1(hidden_states)
        # [Batch_Size, Num_Patches, Embed_Dim] -> [Batch_Size, Num_Patches, Embed_Dim]
        hidde

In [10]:
# the Encoder layer that wraps the above in many layers
print(inspect.getsource(SiglipEncoder))

class SiglipEncoder(nn.Module):
    def __init__(self, config: SiglipVisionConfig):
        super().__init__()
        self.config = config
        self.layers = nn.ModuleList(
            [SiglipEncoderLayer(config) for _ in range(config.num_hidden_layers)]
        )

    def forward(self, inputs_embeds: torch.Tensor) -> torch.Tensor:
        # inputs_embeds: [Batch_Size, Num_Patches, Embed_Dim]
        hidden_states = inputs_embeds

        for encoder_layer in self.layers:
            # [Batch_Size, Num_Patches, Embed_Dim] -> [Batch_Size, Num_Patches, Embed_Dim]
            hidden_states = encoder_layer(hidden_states)

        return hidden_states



In [11]:
## the full vision transformer
print(inspect.getsource(SiglipVisionTransformer))

class SiglipVisionTransformer(nn.Module):

    def __init__(self, config: SiglipVisionConfig):
        super().__init__()
        self.config = config
        self.embed_dim = config.hidden_size

        self.embeddings = SiglipVisionEmbeddings(config)  # equivalent to PatchEmbedding in our vit.py
        self.encoder = SiglipEncoder(config)  # equivalent to TransformerEncoder in our vit.py
        self.post_layernorm = nn.LayerNorm(self.embed_dim, eps=config.layer_norm_eps)

    def forward(self, pixel_values: torch.Tensor) -> torch.Tensor:
        # pixel_values: [Batch_Size, Channels, H, W] -> [Batch_Size, Num_Patches, Embed_Size]
        hidden_states = self.embeddings(pixel_values)
        last_hidden_state = self.encoder(inputs_embeds=hidden_states)
        last_hidden_state = self.post_layernorm(last_hidden_state)
        return last_hidden_state



In [12]:
# the vision model so we can wrap into siglip
print(inspect.getsource(SiglipVisionModel))

class SiglipVisionModel(nn.Module):

    def __init__(self, config: SiglipVisionConfig):
        super().__init__()
        self.config = config
        self.vision_model = SiglipVisionTransformer(config)

    def forward(self, pixel_values) -> Tuple:
        # [Batch_Size, Channels, H, W] -> [Batch_Size, Num_Patches, Embed_Dim]
        return self.vision_model(pixel_values=pixel_values)



## and the full model definition:

In [4]:
# Or instantiate and print the model structure                                                                                   
config = SiglipVisionConfig()
model = SiglipVisionModel(config)                                                                                                
print(model) 

SiglipVisionModel(
  (vision_model): SiglipVisionTransformer(
    (embeddings): SiglipVisionEmbeddings(
      (patch_embedding): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16), padding=valid)
      (position_embedding): Embedding(196, 768)
    )
    (encoder): SiglipEncoder(
      (layers): ModuleList(
        (0-11): 12 x SiglipEncoderLayer(
          (self_attn): SiglipAttention(
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (k_proj): Linear(in_features=768, out_features=768, bias=True)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (layer_norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
          (mlp): SiglipMLP(
            (fc1): Linear(in_features=768, out_features=3072, bias=True)
            (fc2): Linear(in_features=3072, out_features=768, bias=True)
          )
          (layer_norm2): Layer